# Recursion and search

The rest of what functions can do, functions that call themselves, and how much work a program does.

# Part 1: More about functions

## Where Python looks for a name

From week 2: a function can read a name from outside, and a name it assigns to is local. A `def` can also sit inside another `def`.

**Predict:** what are the three lines?

In [ ]:
x = "global"


def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print("inner: ", x)

    inner()
    print("outer: ", x)


outer()
print("main: ", x)

**Predict:** now `inner` doesn't assign to `y`.

In [ ]:
y = "global"


def outer():
    y = "enclosing"

    def inner():
        print("inner: ", y)

    inner()


outer()

Python looks for a name in four places, in this order, and takes the first one it finds:

1. **local**: the function that's running
2. **enclosing**: the functions around it
3. **global**: the top level of the file
4. **built-in**: what Python brings along, such as `print` and `len`

**Predict:**

In [ ]:
def describe(numbers):
    max = numbers[0]
    print(max(numbers))


describe([3, 8, 5])

In [ ]:
print(max(3, 8, 5))

Inside `describe`, the local `max` is a number, and it hides the built-in function: `TypeError: 'int' object is not callable`. Outside, nothing has changed. If you see that error, look for a name like `max`, `list` or `sum` that you've assigned to.

## When Python looks

**Predict:**

In [ ]:
def show():
    print(word)


word = "first"
show()

In [ ]:
word = "second"
show()

`word` didn't exist yet when `def` ran, and that's fine: Python looks a name up when the line **runs**, so it uses the value at that moment.

## Assigning to a name from outside

Week 2's `go_up` failed, because assigning made `level` local. `global` says: this name is the one at the top level.

**Predict:**

In [ ]:
level = 3


def go_up():
    global level
    level = level + 1


go_up()
print(level)

**Predict:** no `global` this time.

In [ ]:
levels = [3]


def go_up_in_list():
    levels[0] = levels[0] + 1


go_up_in_list()
print(levels)

`levels[0] = ...` changes what's **inside** the list. It doesn't assign to the name `levels`, so `levels` stays the outside name and needs no `global`.

`global` works, and you should rarely use it: nobody can tell from a call what it changes. The advice from week 2 stands: pass the value in and `return` the new one.

## A function without brackets

In [ ]:
def shout(text):
    return text.upper() + "!"


print(shout("hi"))

**Predict:** what does this print?

In [ ]:
print(shout)

**Predict:** what does this print?

In [ ]:
loud = shout
print(loud("hey"))

`shout("hi")` **calls** the function. `shout` on its own is the function itself, a value like a number or a list: `print` shows something like `<function shout at 0x...>`. `loud = shout` sticks a second name on the same function (names are labels, as on day 1), so `loud("hey")` calls it.

## What a function carries with it

In [ ]:
def greet(name, greeting="Hello"):
    """Return a greeting for name."""
    message = greeting + ", " + name + "!"
    return message


say = greet

**Predict**:

In [ ]:
say.__name__

In [ ]:
say.__doc__

`say` is only a second label. The function was created as `greet`, and it keeps that name. A function is an object with things attached, and you can read them without calling it:

- `__name__`: the name from the `def`
- `__doc__`: the docstring (this is where `help` gets it, week 2)

Names with double underscores belong to Python itself: you read them, you don't invent your own. Tools read this kind of information too: FastAPI, later in the course, looks at the parameters of your function to decide what a request may contain.

## Passing a function to a function

**Predict:**

In [ ]:
def twice(function, value):
    return function(function(value))


print(twice(shout, "hi"))

**Predict**:

In [ ]:
print(twice(len, "hello"))

`twice` doesn't know which function it gets; it calls whatever it was given. `twice(len, "hello")` is `len(len("hello"))`, which is `len(5)`: a number has no length, so that's a `TypeError`.

## Sorting by something else

**Predict:** the order of each list.

In [ ]:
words = ["pear", "Fig", "banana", "kiwi"]
print(sorted(words))

In [ ]:
print(sorted(words, key=str.lower))

In [ ]:
print(sorted(words, key=len))

- `sorted` without a key compares the strings themselves, character by character, and every uppercase letter comes before every lowercase one: `"Fig"` first.
- `key=len` means: call `len` on every element and sort by **that**. Elements with the same key keep their order (`"pear"` before `"kiwi"`).
- `key=str.lower` sorts as if everything were lowercase, but the list still holds the original strings.

The key is a function, passed without brackets: `sorted` calls it, once per element.

## A function without a name: `lambda`

**Predict:**

In [ ]:
people = [("Ada", 36), ("Grace", 33), ("Alan", 41)]

print(sorted(people))

In [ ]:
("Ada", 36) <= ("Grace", 33)

**Predict**:

In [ ]:
# so how can I sort this by the second element of the tuple?

`lambda person: person[1]` is a small function written in place: it takes `person` and returns `person[1]`. It is the same as

```python
def age(person):
    return person[1]
```

but without a name, for when you need it exactly once. (Its `__name__` is literally `'<lambda>'`, and that's what an error message inside one shows.) A `lambda` holds a single expression, no statements, no `return`.

`max` and `min` take a `key` too: `max(people, key=...)` returns the **person** with the largest key, not the largest key.

**Predict:**

In [ ]:
prices = {"apple": 0.5, "melon": 2.0, "kiwi": 0.3}
print(min(prices))

In [ ]:
print(min(prices, key=lambda name: prices[name]))

In [ ]:
print(min(prices.values()))

Looping over a dict gives its keys, so `min(prices)` is the alphabetically first **name**. With the key, it's the name whose **price** is lowest. `min(prices.values())` is the lowest price itself, but then you no longer know whose it is.

## Stars in the parameter list

**Predict:** what does each call print?

In [ ]:
def show_all(first, *rest, **options):
    print(first, rest, options)


show_all(1)

In [ ]:
show_all(1, 2, 3)

In [ ]:
show_all(1, 2, colour="red", size=3)

In [ ]:
show_all()

- `*rest` collects all further positional arguments into a **tuple** (empty if there are none).
- `**options` collects all further keyword arguments into a **dict**.
- `first` is an ordinary parameter, so `show_all()` is a `TypeError`.

Only the stars matter; the usual names are `*args` and `**kwargs`. The order is ordinary parameters first, then `*args`, then `**kwargs`. This is how `print(1, 2, 3, sep="-")` and `max(3, 8, 5)` take any number of arguments.

The stars also work the other way round, in a **call**. **Predict:**

In [ ]:
def add_three(a, b, c):
    return a + b + c


numbers = [1, 2, 3]
print(add_three(*numbers))

In [ ]:
print(add_three(numbers))

In [ ]:
settings = {"b": 20, "c": 30}
print(add_three(1, **settings))

In a call, `*numbers` unpacks the list into separate positional arguments, and `**settings` unpacks the dict into keyword arguments: `add_three(1, b=20, c=30)`. Without the star, the list is **one** argument, and `b` and `c` are missing.

Put together with functions as values. **Predict:**

In [ ]:
def logged(function, *args, **kwargs):
    print("calling", function.__name__, "with", args, kwargs)
    return function(*args, **kwargs)


print(logged(max, 3, 8, 5))

In [ ]:
print(logged(greet, "Ada", greeting="Hi"))

`logged` doesn't need to know what `function` expects: it collects whatever arrives and passes it on unchanged. Forwarding like this is the main use of `*args` and `**kwargs`.

In your own functions, prefer named parameters where you can. `def area(width, height)` tells the reader what it wants; `def area(*args)` doesn't, and a wrong call fails later or not at all. In a review, it's worth asking why a function needs the stars.

## A function inside a function

A function is a value, so a function can also **return** one. **Predict:**

In [ ]:
def make_adder(n):
    def adder(x):
        return x + n

    return adder


add_5 = make_adder(5)
print(add_5(3))

In [ ]:
add_8 = make_adder(8)
print(add_8(3))

In [ ]:
print(add_5(3))

`make_adder` has finished by the time `add_5` is called, but `adder` still sees its `n`: the enclosing scope stays around for as long as the inner function needs it. Each call of `make_adder` has its own `n`. A function together with the names it keeps from its enclosing scope is called a **closure**.

**Predict:**

In [ ]:
def make_counter():
    count = 0

    def counter():
        count = count + 1
        return count

    return counter


tick = make_counter()
print(tick())

The same rule as `go_up`: assigning makes `count` local to `counter`, and it has no value yet. `global` wouldn't help, because `count` isn't at the top level. For an enclosing function, the word is `nonlocal`.

**Predict:**

In [ ]:
def make_counter():
    count = 0

    def counter():
        nonlocal count
        count = count + 1
        return count

    return counter


tick = make_counter()
print(tick())

In [ ]:
print(tick())

In [ ]:
tock = make_counter()
print(tock())

In [ ]:
print(tick())

`tick` and `tock` each have their own `count`, kept between calls. Next week, classes give you a clearer way to write state like this.

## All of it at once

**Predict:** the four lines.

In [ ]:
def scope_test():
    def do_local():
        spam = "local spam"

    def do_nonlocal():
        nonlocal spam
        spam = "nonlocal spam"

    def do_global():
        global spam
        spam = "global spam"

    spam = "test spam"
    do_local()
    print("After local assignment:", spam)
    do_nonlocal()
    print("After nonlocal assignment:", spam)
    do_global()
    print("After global assignment:", spam)


scope_test()
print("In global scope:", spam)

`do_local` makes its own `spam`, which disappears when it returns. `do_nonlocal` changes the one in `scope_test`. `do_global` creates one at the top level, which `scope_test` doesn't see, because it finds its own first.

# Part 2: Recursion

## A function that calls itself

**Predict:** what does this print?

In [ ]:
def countdown(n):
    if n == 0:
        print("go!")
    else:
        print(n)
        countdown(n - 1)


countdown(3)

A function may call itself. Each call works on a smaller `n` until `n == 0`, which doesn't call again. Two parts, always:

- a **base case** that answers without calling itself (`n == 0`),
- a **recursive case** that calls itself with a smaller problem (`n - 1`).

**Predict:**

In [ ]:
def mystery(n):
    if n > 0:
        mystery(n - 1)
    print(n)


mystery(3)

The `print` comes **after** the call, so it waits until that call is finished. Each call has its own frame with its own `n` (as in week 2: every call gets its own local names):

```
mystery(3)            n = 3: calls mystery(2), then prints 3
  mystery(2)          n = 2: calls mystery(1), then prints 2
    mystery(1)        n = 1: calls mystery(0), then prints 1
      mystery(0)      n = 0: makes no call, prints 0
```

`mystery(0)` is the only call that doesn't wait for another one, so it prints first: 0, 1, 2, 3.

## Returning a value

**Predict:** `total([4, 5, 6])`

In [ ]:
def total(numbers):
    if numbers == []:
        return 0
    return numbers[0] + total(numbers[1:])


print(total([4, 5, 6]))

In [ ]:
print(total([]))

The idea: the total of a list is its first number plus the total of the rest. The rest is shorter, so eventually it's `[]`, whose total is 0.

```
total([4, 5, 6]) = 4 + total([5, 6])
                 = 4 + (5 + total([6]))
                 = 4 + (5 + (6 + total([])))
                 = 4 + (5 + (6 + 0))   = 15
```

(For a list, `sum` does this in one call. Recursion pays off where a loop doesn't fit as well, as with the nested data further down.)

## One more function

**Predict:**

In [ ]:
def count_up(n):
    return count_up(n + 1)


count_up(0)

Every call waits for the next, so the frames pile up. Python stops at about 1000 of them with a `RecursionError` ("maximum recursion depth exceeded"). That's almost always a missing or unreachable base case.

**Predict:** what does each of the three calls do?

In [ ]:
def half_down(n):
    if n == 0:
        return "done"
    return half_down(n // 2)


print(half_down(0))

In [ ]:
print(half_down(1000))

In [ ]:
print(half_down(-1))

`1000 // 2` is 500, then 250, ..., 1, 0: done. But `-1 // 2` is `-1` (`//` rounds **down**), so `half_down(-1)` calls `half_down(-1)` again, forever. The base case is never reached for negative numbers: the recursive case has to get **closer** to the base case on every call, for every input.

## Recursion on nested data

Exercise 3's file system, written as dicts inside dicts: a file's value is its size, a directory's value is another dict.

In [ ]:
tree = {
    "notes.txt": 100,
    "photos": {
        "trip.jpg": 4000,
        "summer": {"beach.jpg": 2500, "sunset.jpg": 1500},
    },
    "music": {"song.mp3": 3000},
}

In [ ]:
def size(directory):
    result = 0
    for name, value in directory.items():
        if isinstance(value, dict):  # asks: is `value` a dict?
            result += size(value)
        else:
            result += value
    return result

**Predict:**

In [ ]:
print(size(tree))

In [ ]:
print(size(tree["photos"]))

11100 and 8000, the same totals as in Exercise 3, with no paths and no `cd ..`. The function doesn't need to know how deep the tree goes: each call handles one directory and leaves the subdirectories to calls of itself. A tree is the textbook case for recursion, because a part of a tree is again a tree.

**Predict:** what does it print?

In [ ]:
def show(directory, indent):
    for name, value in directory.items():
        if isinstance(value, dict):
            print(indent + name + "/")
            show(value, indent + "    ")
        else:
            print(indent + name, value)


show(tree, "")

# Part 3: How much work?

## Counting ways up the stairs

How many ways are there up a staircase of `n` stairs, taking 1 or 2 stairs at a time? With 0 or 1 stairs, there is exactly one way. Otherwise, the first move takes 1 stair (and `n - 1` are left) or 2 stairs (`n - 2` are left).

In [ ]:
def ways(n):
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)

**Predict:** `ways(2)` and `ways(4)`. Write the ways down.

In [ ]:
print(ways(2))

In [ ]:
print(ways(4))

In [ ]:
for n in range(1, 11):
    print(n, ways(n))

## Counting calls

How much work was that? A dict outside the function counts the calls. (The function changes what's **inside** the dict; it never assigns to the name `calls`, so this works, like `levels` in Part 1.)

In [ ]:
calls = {"count": 0}


def ways_counted(n):
    calls["count"] += 1
    if n <= 1:
        return 1
    return ways_counted(n - 1) + ways_counted(n - 2)

**Predict:** roughly how does the number of calls change when `n` grows by 1?

In [ ]:
for n in range(2, 16):
    calls["count"] = 0
    ways_counted(n)
    print(n, calls["count"])

Each step up in `n` makes the work roughly **1.6 times** as large. That doesn't sound like much. Let's time it:

In [ ]:
import time

for n in [30, 32, 34, 36]:
    start = time.perf_counter()
    ways(n)
    print(n, round(time.perf_counter() - start, 3), "seconds")

**Predict:** if this pattern goes on, how long would `n = 50` take? And `n = 100`?

(Don't run them. Work it out from the timings.)

In [ ]:
print(1.6 ** (50 - 36))

In [ ]:
print(1.6 ** (100 - 36))

Those are the factors on top of the time for `n = 36`, about a second on a typical laptop: roughly a quarter of an hour for `n = 50`, and hundreds of thousands of years for `n = 100`. A hundred stairs isn't a big input, and a faster computer doesn't help much.

How the work grows with the size of the input matters more than how fast one step is:

| work grows like | n = 10 | n = 100 |
|---|---|---|
| n (one loop) | 10 | 100 |
| n² (a loop in a loop) | 100 | 10,000 |
| about ×1.6 for each step in n (`ways`) | 177 calls | about 10²¹ calls |

# Part 4: Grids

## A grid as a list of lists

**Predict:**

In [ ]:
rows = ["######", "#S.#.#", "#..#.#", "##..G#", "######"]
grid = [list(row) for row in rows]

In [ ]:
for line in grid:
    print("".join(line))

In [ ]:
print(grid[3][4])

In [ ]:
print(len(grid), len(grid[0]))

`grid[row][col]`: first pick the row, then the position in it. `len(grid)` is the number of rows, `len(grid[0])` the number of columns.